# Clinical Evidence Researcher Agent with Strands
This notebook demonstrates how to build a clinical evidence researcher agent using the open-source [Strands Agents](https://strandsagents.com/latest/) framework and how to deploy the agent on [Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/).

The agent will need the following tools:

- `query_pubmed`: Searches PubMed API for biomedical literature
- `retrieve`: Retrieves internal evidence from the NCBI knowledge base hosted on Amazon Bedrock

#### Install Strands agents and required dependencies

In [ ]:
%pip install strands-agents strands-agents-tools xmltodict --quiet

#### Ensure the latest version of boto3 is shown below
Ensure the boto3 version printed below is **1.37.1** or higher.

In [ ]:
%pip show boto3

#### Import required libraries

In [ ]:
import os
import boto3
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import retrieve

from utils.PubMed import PubMed

# Initialize KB tool variable
kb_tool = None

# Get AWS account information
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = boto3.Session().region_name

# Define Bedrock model id
MODEL_ID = "global.anthropic.claude-sonnet-4-20250514-v1:0"

## Prerequisites

Run through the notebook environment setup in [00-setup_environment.ipynb](00-setup_environment.ipynb).

#### Setup AWS clients
Define the clients to AWS services that will be used by tools.

In [ ]:
# Initialize AWS clients
bedrock_client = boto3.client('bedrock-runtime', region_name=region)
bedrock_agent_client = boto3.client("bedrock-agent", region_name=region)

print(f"Region: {region}")
print(f"Account ID: {account_id}")

#### Setup Knowledge Base for internal evidence retrieval
In this example we are going to use a built-in tool from Strands agents called **`retrieve`**. This tool semantically retrieve data from Amazon Bedrock Knowledge Bases for RAG, memory, and other purposes. The tool requires the knowledge base id, which we'll be provided throught the environment variable `KNOWLEDGE_BASE_ID` defined below.

In [ ]:
# Find the Knowledge Base
response = bedrock_agent_client.list_knowledge_bases()

# Iterate through knowledge bases and find needed one
ncbi_kb_id = None
for kb in response['knowledgeBaseSummaries']:
    kb_name = kb['name']
    if 'ncbiKnowledgebase' in kb_name:
        ncbi_kb_id = kb['knowledgeBaseId']
        break

if ncbi_kb_id:
    print(f"Found Knowledge Base ID: {ncbi_kb_id}")
    os.environ["KNOWLEDGE_BASE_ID"] = ncbi_kb_id
    print("Knowledge Base will be integrated using direct Strands tool approach")
else:
    print("Warning: Knowledge Base not found. Internal evidence retrieval may not work.")
    ncbi_kb_id = None

## Strands Agent Creation
In this section we create the agent using the Strands framework

#### Define agent configuration and instructions

In [ ]:
clinical_research_agent_name = "Clinical-evidence-researcher-strands"
clinical_research_agent_description = "Research internal and external evidence using Strands framework"
clinical_research_agent_instruction = """You are a medical research assistant AI specialized in summarizing internal and external 
evidence related to cancer biomarkers. Your primary task is to interpret user queries, gather internal and external 
evidence, and provide relevant medical insights based on the results. Use only the appropriate tools as required by 
the specific question. Always use the retrieve knowledge base tool first for internal evidence search. Follow these instructions carefully: 

1. Use the retrieve tool to search internal evidence. Use the query PubMed tool after you performed a search using the retrieve tool.

2. When querying PubMed: 
   a. Summarize the findings of each relevant study with citations to the specific pubmed web link of the study 
   b. The json output will include 'Link', 'Title', 'Summary'. 
   c. Always return the Title and Link (for example, 'https://pubmed.ncbi.nlm.nih.gov/') of each study in your response.  

3. For internal evidence, make use of the knowledge base to retrieve relevant information. 
   Always provide citations to specific content chunks. 

4. When providing your response: 
   a. Start with a brief summary of your understanding of the user's query. 
   b. Explain the steps you're taking to address the query. Ask for clarifications from the user if required. 
   c. Separate the responses generated from internal evidence (knowledge base) and external evidence (PubMed api).  
   d. Conclude with a concise summary of the findings and their potential implications for medical research.
"""

#### Define tools for Strands agent
We are going to use a custom tool to query PubMed combined with the retrieve tool from Strands framework. Retrieve tool doesn't need to be defined by a function like custom tools, so we simply add it to the list of tools.

In [ ]:
# Define the tools using Strands @tool decorator
@tool
def query_pubmed(query: str) -> str:
    """
    Query PubMed for relevant biomedical literature based on the user's query.
    This tool searches PubMed abstracts and returns relevant studies with titles, links, and summaries.
    
    Args:
        query (str): The search query for PubMed
    
    Returns:
        str: JSON string containing PubMed search results with titles, links, and summaries
    """
    
    pubmed = PubMed()

    print(f"\nPubMed Query: {query}\n")
    result = pubmed.run(query)
    print(f"\nPubMed Results: {result}\n")
    return result

# Create list of custom tools
clinical_research_agent_tools = [query_pubmed, retrieve]
print(f"Created {len(clinical_research_agent_tools)} custom tools for the Strands agent")

#### Setup AWS Bedrock provider for Strands

In [ ]:
# Create Bedrock model for Strands
model = BedrockModel(
    model_id=MODEL_ID,
    region_name=region,
    temperature=0.1,
    streaming=True
)

#### Create the Strands agent

In [ ]:
# Create the Strands agent
try:
    # Use the custom tools we created
    clinical_evidence_agent = Agent(
        model=model,
        tools=clinical_research_agent_tools,
        system_prompt=clinical_research_agent_instruction
    )
    
    print(f"Successfully created Strands agent: {clinical_research_agent_name}")
    print(f"Agent has {len(clinical_research_agent_tools)} tools available:")
    for tool in clinical_research_agent_tools:
        print(f"  - {tool.__name__}")
    
except Exception as e:
    print(f"Error creating agent: {e}")
    raise

#### Test the agent locally

In [ ]:
# Test the agent with a research query
test_query = "Can you search PubMed for evidence around the effects of biomarker use in oncology on clinical trial failure risk"

print(f"Testing agent with query: {test_query}")
print("=" * (26 + len(test_query)))

try:
    # Run the agent
    response = clinical_evidence_agent(test_query)
    
except Exception as e:
    print(f"Error during agent execution: {e}")
    import traceback
    traceback.print_exc()

#### Advanced usage examples

In [ ]:
# Example of more complex queries
complex_queries = [
    "Search for evidence on LRIG1 biomarker in lung cancer prognosis",
    "Find studies about biomarker-guided therapy in precision oncology",
    "What does the internal knowledge base say about molecular phenotypes and imaging?"
]

for test_query in complex_queries: 
    print(f"\nTesting query: {test_query}")
    print("=" * (15 + len(test_query)))
    
    try:
        response = clinical_evidence_agent(test_query)
    except Exception as e:
        print(f"Error: {e}")

#### Session management and conversation continuity

In [ ]:
# Demonstrate conversation continuity
def interactive_research_session():
    """
    Simple interactive session with the clinical evidence researcher agent
    """
    print("Interactive Clinical Evidence Research Session")
    print("Ask about biomarkers, clinical trials, or cancer research")
    print("Type 'quit' to exit")
    print("=" * 60)
    
    while True:
        user_input = input("\nYour research question: ")
        
        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Research session ended.")
            break
            
        try:
            response = clinical_evidence_agent(user_input)                
        except Exception as e:
            print(f"Error: {e}")

interactive_research_session()

## Deploy agent on AgentCore
In this section we are going to deploy the clinical evidence researcher agent to [Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) Runtime.

### Preparing your agent for deployment on AgentCore Runtime

In [ ]:
%%writefile clinical_research_agent_runtime.py

import json
import uuid
import strands
from strands import Agent
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from clinical_research_agent import *

# Create the Strands agent
try:
    agent = Agent(
        model=model,
        tools=clinical_research_agent_tools,
        system_prompt=clinical_research_agent_instruction
    )
    
    print("Successfully created Strands agent")
    print(f"Agent has {len(clinical_research_agent_tools)} tools available")

except Exception as e:
    print(f"Error creating agent: {e}")
    raise

# Custom event serializer function
def json_serializer(obj):
    if isinstance(obj, strands.agent.agent.Agent):
        return obj.name
    elif isinstance(obj, strands.agent.agent_result.AgentResult):
        result = {}
        result["message"] = str(obj)
        result["metrics"] = {}
        result["metrics"]["accumulated_usage"] = obj.metrics.accumulated_usage
        result["metrics"]["accumulated_metrics"] = obj.metrics.accumulated_metrics
        return result
    elif isinstance(obj, uuid.UUID):
        return str(obj)
    else:
        # Assume other objects are not serializable
        return None

app = BedrockAgentCoreApp()

@app.entrypoint
async def strands_agent_bedrock_streaming(payload):
    """
    Invoke the agent with streaming capabilities
    This function demonstrates how to implement streaming responses
    with AgentCore Runtime using async generators
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    
    try:
        # Stream each chunk as it becomes available
        async for event in agent.stream_async(user_input):
            print("Received event:", event)
            # serialize the event object
            yield json.dumps(event, default=json_serializer)
    except Exception as e:
        # Handle errors gracefully in streaming context
        error_response = {"error": str(e), "type": "stream_error"}
        print(f"Streaming error: {error_response}")
        yield error_response

if __name__ == "__main__":
    app.run()

### Deploying the agent to AgentCore Runtime

#### Define agent name and retrieve runtime role

In [ ]:
from utils.boto3_helper import get_role_arn
iam = boto3.client('iam')

agent_name="clinical_research_agent"
agentcore_iam_role = get_role_arn('BedrockAgentCoreStrands')
agentcore_iam_role

#### Configure AgentCore Runtime deployment
During the configure step, your docker file will be generated based on your application code.

In [ ]:
import os
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

# Remove existing Dockerfile so configure() regenerates it with the correct entrypoint
if os.path.exists("Dockerfile"):
    os.remove("Dockerfile")

agentcore_runtime = Runtime()

response = agentcore_runtime.configure(
    entrypoint="clinical_research_agent_runtime.py",
    execution_role=agentcore_iam_role,
    auto_create_ecr=True,
    requirements_file="runtime_requirements.txt",
    region=region,
    agent_name=agent_name
)
response

#### Launching agent to AgentCore Runtime
Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime.

In [ ]:
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True
)
launch_result

### Invoking AgentCore Runtime with boto3
Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import json
import boto3
from IPython.display import Markdown, display

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

test_query = "Search for evidence on LRIG1 biomarker in lung cancer prognosis"
response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": test_query})
)

print(f"Response contentType: {response.get("contentType")}\n")
print(f"Testing agent: {test_query}")
print("=" * (15 + len(test_query)))

if "text/event-stream" in response.get("contentType", ""):
    # Processing streaming response
    for line in response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                # remove the SSE structure
                data = line[6:]
                # we need to parse it twice to convert from JSON str to a dictionary
                data_obj = json.loads(data)
                data_obj = json.loads(data_obj)
                # for this example we only care about the data field
                if "data" in data_obj:
                    print(data_obj.get("data"), end="", flush=True)
    print()  # final newline after streaming completes
else:
    # Handle non-streaming response
    try:
        events = []
        for event in response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    if events:
        try:
            response_data = json.loads(events[0].decode("utf-8"))
            display(Markdown(response_data))
        except:
            print(f"Raw response: {events[0]}")